In [10]:
!pip install -q groq pymupdf pillow


In [11]:
import json
import fitz
import base64
from PIL import Image
from io import BytesIO
from kaggle_secrets import UserSecretsClient
from groq import Groq

# Load Groq API Key securely
user_secrets = UserSecretsClient()
groq_api_key = user_secrets.get_secret("GROQ_API_KEY")

client = Groq(api_key=groq_api_key)


In [12]:
def pdf_to_images(pdf_path, dpi=200):
    doc = fitz.open(pdf_path)
    return [Image.frombytes("RGB", [page.get_pixmap(dpi=dpi).width, page.get_pixmap(dpi=dpi).height],
                            page.get_pixmap(dpi=dpi).samples) for page in doc]


In [13]:
def image_to_base64(image):
    buffer = BytesIO()
    image.save(buffer, format="JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


In [18]:
def extract_text_with_vision(image):
    base64_image = image_to_base64(image)

    vision_prompt = """
Extract all readable text from this document.
Preserve layout and line breaks.
Do NOT summarize.
Return only raw extracted text.
"""

    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}},
                {"type": "text", "text": vision_prompt}
            ],
        }],
        temperature=0
    )

    return response.choices[0].message.content.strip()



In [19]:
def extract_invoice_fields(text):

    prompt = f"""
Extract structured invoice data from the text below. Do not explain or hallucinate

Use semantic reasoning to match fields even if formatting is inconsistent.

Return exactly the following JSON structure with values or null if missing:

{{
"Vendor":{{"BusinessName":null,"Address":null,"GSTIN":null,"PAN":null,"Phone":null,"Email":null,"CIN":null}},
"Buyer":{{"Name":null,"BillingAddress":null,"ShippingAddress":null,"GSTIN":null,"Phone":null,"Email":null}},
"Items":[{{"Description":null,"Quantity":null,"Unit":null,"RatePerUnit":null,"Discount":null,"TaxableValue":null,
"GSTRatePercent":null,"CGSTAmount":null,"SGSTAmount":null,"IGSTAmount":null,"Cess":null,"TotalItemAmount":null}}],
"Totals":{{"Subtotal":null,"TotalTaxableValue":null,"TotalCGST":null,"TotalSGST":null,"TotalIGST":null,"TotalCess":null,
"RoundOff":null,"GrandTotal":null,"AmountInWords":null}},
"PaymentDetails":{{"ModeOfPayment":null,"UPIID":null,"BankName":null,"AccountNumber":null,"IFSCCode":null,
"TransactionReferenceID":null}}
}}

TEXT:
{text}
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip()


In [20]:
def process_llm_output(llm_output, threshold=0.6):

    try:
        data = json.loads(llm_output)
    except:
        print("Invalid JSON returned:")
        print(llm_output)
        return None

    # Recursively nullify low-confidence fields (if confidence keys exist)
    def sanitize(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if isinstance(v, dict) and "confidence" in v:
                    if v["confidence"] < threshold:
                        v["value"] = None
                else:
                    sanitize(v)
        elif isinstance(obj, list):
            for item in obj:
                sanitize(item)

    sanitize(data)
    return data


In [33]:
def process_document(file_path):

    full_text = ""

    if file_path.lower().endswith(".pdf"):
        images = pdf_to_images(file_path)
        for img in images:
            full_text += extract_text_with_vision(img) + "\n"
    else:
        image = Image.open(file_path).convert("RGB")
        full_text = extract_text_with_vision(image)
    
    print(full_text)
    print("=======================================================")
    llm_output = extract_invoice_fields(full_text)
    structured_data = process_llm_output(llm_output)

    # 🔥 Only print JSON
    print(json.dumps(structured_data, indent=2))

    return structured_data


In [34]:
result = process_document("/kaggle/input/datasets/nivetha20042005/more-fields/sample_indian_gst_invoice.pdf")


TAX INVOICE

Seller: ABC Technologies Pvt. Ltd.
Registered Office: 4th Floor, Tech Park, MG Road, Bengaluru, Karnataka - 560001
GSTIN: 29ABCDE1234F1Z5
PAN: ABCDE1234F
State Code: 29
CIN: U12345KA2020PTC123456
Email: accounts@abctech.in | Phone: +91-80-12345678

Bill To: XYZ Retailers LLP
Address: 21 Market Street, T Nagar, Chennai, Tamil Nadu - 600017
GSTIN: 33XYZAB5678L1Z2
PAN: XYZAB5678L
State Code: 33
Place of Supply: Tamil Nadu (33)

Invoice No: INV/2026-27/0045
Invoice Date: 11-Feb-2026
Due Date: 26-Feb-2026
Purchase Order No: PO-7788
Reverse Charge: No
E-Way Bill No: 181234567890
IRN: 7f9c2e6d8a1234567890abcdef1234567890abcdef123456

|   | HSN/SAC | Qty | Unit | Rate (₹) | Taxable Value (₹) | CGST 9% (₹) | SGST 9% (₹) | Total (₹) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
|   | 998314 | 5 | Nos | 15000 | 75000 | 6750 | 6750 | 88500 |
| (AMC) | 998717 | 1 | Year | 25000 | 25000 | 2250 | 2250 | 29500 |
|   | 998316 | 12 | Months | 2000 | 24000 | 2160 | 2160 | 28320 |

In [35]:
result = process_document("/kaggle/input/datasets/rithankoushik/invoice-sample/Invoice_Corrected.pdf")


TAX INVOICE

ABC Technologies Pvt. Ltd.
4th Floor, Tech Park, MG Road, Bengaluru, Karnataka - 560001
Email: accounts@abctech.in
Phone: +91-80-12345678
CIN: U12345KA2020PTC123456
GSTIN:             29ABCDE1234F1Z5

STATUS: PENDING

Invoice No:        INV-2026-001
Date:              Feb 16, 2026
Due Date:          Mar 16, 2026


BILL TO (BUYER)

Name:              XYZ Retailers LLP
Address:           21 Market Street, T Nagar, Chennai, Tamil Nadu - 600017
GSTIN:             33XYZAB5678L1Z2


| # | Description                | Qty    | Rate   | Taxable | CGST  | SGST  | Total  |
|----|---------------------------|--------|--------|---------|-------|-------|--------|
| 1  | (AMC) Annual Maintenance  | 1 Year | 25,000 | 25,000  | 2,250 | 2,250 | 29,500 |
| 2  | Support Services (Monthly) | 12 Months | 2,000  | 24,000  | 2,160 | 2,160 | 28,320 |
| 3  | Hardware Modules           | 5 Nos  | 15,000 | 75,000  | 6,750 | 6,750 | 88,500 |

Amount in Words:
Rupees One Lakh Forty Six Thousand Three H

In [27]:
result = process_document("/kaggle/input/datasets/rithankoushik/invoice-sample/handinvoice.jpeg")


INVOICE  
SAMSUDDIN SIDDIQUI  
PAN No. DODPS7841F  

Specialist in : Marbles, Tiles, Polishing, Ladi Polish, Vertical Blinds, Carpet - Sofa Cleaning, Curtain Furniture, Pop, Revolving Chair, Painting, Aluminium Works All Types of Chair Repairing Etc.  

Ekta Nagar, Azad Nagar, B.H. Khalil Seth Godown, Ghatkopar (W), Mumbai - 400 086.  

M/s. BVC Logistics. Pvt. Ltd.  
Plot. No-35 Sukrut Building. Tarm Bharat -CHS.  
Ltd - J.B. Nagar - Andheri(E) Mumbai  

Sr. No   PARTICULARS   QTY PCS.   SIZE   RATE   AMOUNT Rs.   P.  

1)   Falash Tank Rush Button - 1  240  240 - 00  
2)   Zed. Exp. Brash. - 1  550  550 - 00  
3)   Bathrooms comet English New. Fitting. 2  230  460 - 00  
4)   Preshar Nat. uminarcnas 2  450  900 - 00  
5)   Pipe Zed. Exp.and Preshor Pipe 2  195  390 - 00  
6)   Plomber charge. -  2300  2300 - 00  
7)   Pata Bolt - New - 1  836  836 - 00  
8)   Hodrotics gas - 1  750  750 - 00  
9)   Labor charge - - 800  800 - 00  

Rupees in Word : Seven thousand. Two hundred. Twenty

In [ ]:
result = process_document("/kaggle/input/datasets/nivetha20042005/complex-dataset-2/Complex_Invoice_With_Customer_GST_1.pdf")


2.4 sec